# Aneurysm Volume Prediction v2 Refined (Based on 0.7163)

这个版本基于你得分更高的v2继续改进，避免v3那类过拟合配置：
1. 不使用高风险 `epochs=100/base_ch=28/bs=1`
2. 使用体积分层5折(StratifiedKFold on volume bins)
3. epochs=60 + early stopping(patience=12)
4. 保留TTA、OOF阈值搜索、软硬融合、OOF校准
5. 路径保持竞赛平台强适配


In [ ]:
# 如缺包请取消注释安装
# %pip install nibabel scikit-learn tqdm pandas matplotlib
# %pip install torch torchvision torchaudio


In [1]:
import os
import sys
import json
import math
import time
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm
from sklearn.model_selection import KFold, StratifiedKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_AMP = (DEVICE.type == "cuda")
print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("Device:", DEVICE, "AMP:", USE_AMP)


Python: /Users/songling/Documents/conda/miniconda3/envs/aneurysm-seg/bin/python
Torch: 2.10.0
Device: mps AMP: False


In [2]:
# ===== 路径配置：本地/平台/Kaggle 自动适配 =====
# 可选手动指定：
# os.environ["ANEURYSM_DATA_ROOT"] = "/Users/songling/Desktop/Aneurysm Volume Prediction"

DATA_ROOT = os.environ.get("ANEURYSM_DATA_ROOT", "").strip()

def is_valid_dataset_dir(p: Path) -> bool:
    return (p / "train").exists() and (p / "test").exists() and (p / "train_labels").exists()

def find_base_dir() -> Path:
    if DATA_ROOT:
        p = Path(DATA_ROOT)
        if is_valid_dataset_dir(p):
            return p

    fixed = [
        "/dataset/public",
        "/dataset",
        "/kaggle/input/aneurysm-volume-prediction",
        "/kaggle/input/aneurysm-volume",
        "/Users/songling/Desktop/Aneurysm Volume Prediction",
    ]
    for c in fixed:
        p = Path(c)
        if is_valid_dataset_dir(p):
            return p

    for root in [Path("/dataset"), Path("/kaggle/input"), Path.cwd()]:
        if not root.exists():
            continue
        for d in root.rglob("*"):
            if d.is_dir() and is_valid_dataset_dir(d):
                return d
    return None

BASE_DIR = find_base_dir()
if BASE_DIR is None:
    raise FileNotFoundError("未找到数据目录，请设置 ANEURYSM_DATA_ROOT")

TRAIN_IMG_DIR = BASE_DIR / "train"
TRAIN_MASK_DIR = BASE_DIR / "train_labels"
TEST_IMG_DIR = BASE_DIR / "test"
TRAIN_CSV = BASE_DIR / "train.csv"
SAMPLE_SUB_CSV = BASE_DIR / "sample_submission.csv"

# 平台常见输出目录优先级
if Path("/root/setup/solution/working").exists():
    OUTPUT_ROOT = Path("/root/setup/solution/working")
elif Path("/working").exists():
    OUTPUT_ROOT = Path("/working")
elif Path("/kaggle/working").exists():
    OUTPUT_ROOT = Path("/kaggle/working")
else:
    OUTPUT_ROOT = BASE_DIR / "working"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = os.environ.get("RUN_NAME", "").strip() or datetime.now().strftime("segcv_v2_%Y%m%d_%H%M%S")
EXP_DIR = OUTPUT_ROOT / RUN_NAME
CKPT_DIR = EXP_DIR / "checkpoints"
PRED_DIR = EXP_DIR / "predictions"
LOG_DIR = EXP_DIR / "logs"
for d in [EXP_DIR, CKPT_DIR, PRED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXP_DIR:", EXP_DIR)


BASE_DIR: /Users/songling/Desktop/Aneurysm Volume Prediction
OUTPUT_ROOT: /Users/songling/Desktop/Aneurysm Volume Prediction/working
EXP_DIR: /Users/songling/Desktop/Aneurysm Volume Prediction/working/segcv_v2_20260219_224419


In [3]:
def load_nii(path: Path):
    nii = nib.load(str(path))
    arr = nii.get_fdata(dtype=np.float32)
    spacing = nii.header.get_zooms()[:3]
    return arr, spacing

def robust_zscore(x, eps=1e-6):
    lo, hi = np.percentile(x, [0.5, 99.5])
    x = np.clip(x, lo, hi)
    m, s = x.mean(), x.std()
    return (x - m) / (s + eps)

def volume_from_binary_mask(mask_zyx, spacing):
    voxel_vol = float(spacing[0] * spacing[1] * spacing[2])
    return float(max(mask_zyx.sum() * voxel_vol, 0.0))

def volume_from_prob(prob_zyx, spacing):
    voxel_vol = float(spacing[0] * spacing[1] * spacing[2])
    return float(max(prob_zyx.sum() * voxel_vol, 0.0))

def volumetric_similarity(y_true, y_pred, eps=1e-4):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    vs = 1.0 - np.abs(y_true - y_pred) / (y_true + y_pred + eps)
    return float(np.mean(vs))

def parse_pid(path: Path):
    return int(path.name.split(".")[0])


In [5]:
class AneurysmDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, augment=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.augment = augment

    def __len__(self):
        return len(self.img_paths)

    def _aug(self, img, msk):
        # img/msk: (1, D, H, W)
        if random.random() < 0.5:
            img = torch.flip(img, dims=[2])
            if msk is not None:
                msk = torch.flip(msk, dims=[2])
        if random.random() < 0.5:
            img = torch.flip(img, dims=[3])
            if msk is not None:
                msk = torch.flip(msk, dims=[3])

        if random.random() < 0.5:
            k = random.randint(0, 3)
            img = torch.rot90(img, k=k, dims=[2, 3])
            if msk is not None:
                msk = torch.rot90(msk, k=k, dims=[2, 3])

        if random.random() < 0.7:
            scale = 1.0 + random.uniform(-0.15, 0.15)
            shift = random.uniform(-0.15, 0.15)
            gamma = 1.0 + random.uniform(-0.2, 0.2)
            # 先平移到正区间做gamma，再还原
            mn = img.min()
            x = img - mn
            x = x / (x.max() + 1e-6)
            x = x.clamp(0, 1) ** gamma
            img = (x * scale) + shift

        if random.random() < 0.3:
            noise = torch.randn_like(img) * 0.03
            img = img + noise

        return img, msk

    def __getitem__(self, idx):
        p = self.img_paths[idx]
        arr, spacing = load_nii(p)
        arr = robust_zscore(arr)
        # (H,W,D) -> (D,H,W)
        arr = np.transpose(arr, (2, 0, 1)).astype(np.float32)
        img = torch.from_numpy(arr).unsqueeze(0)

        out = {
            "image": img,
            "spacing": torch.tensor(spacing, dtype=torch.float32),
            "pid": parse_pid(p),
        }

        if self.mask_paths is not None:
            m, _ = load_nii(self.mask_paths[idx])
            m = (m > 0.5).astype(np.float32)
            m = np.transpose(m, (2, 0, 1)).astype(np.float32)
            msk = torch.from_numpy(m).unsqueeze(0)
            if self.augment:
                img, msk = self._aug(img, msk)
                out["image"] = img
            out["mask"] = msk
        elif self.augment:
            img, _ = self._aug(img, None)
            out["image"] = img

        return out


In [6]:
class ResBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.proj = nn.Conv3d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        identity = self.proj(x)
        x = self.act(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = self.act(x + identity)
        return x

class ResUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=24):
        super().__init__()
        self.e1 = ResBlock3D(in_ch, base)
        self.p1 = nn.MaxPool3d(2)
        self.e2 = ResBlock3D(base, base*2)
        self.p2 = nn.MaxPool3d(2)
        self.e3 = ResBlock3D(base*2, base*4)
        self.p3 = nn.MaxPool3d(2)
        self.b = ResBlock3D(base*4, base*8)

        self.u3 = nn.ConvTranspose3d(base*8, base*4, 2, stride=2)
        self.d3 = ResBlock3D(base*8, base*4)
        self.u2 = nn.ConvTranspose3d(base*4, base*2, 2, stride=2)
        self.d2 = ResBlock3D(base*4, base*2)
        self.u1 = nn.ConvTranspose3d(base*2, base, 2, stride=2)
        self.d1 = ResBlock3D(base*2, base)
        self.head = nn.Conv3d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))

        d3 = self.u3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.d3(d3)

        d2 = self.u2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.d2(d2)

        d1 = self.u1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.d1(d1)
        return self.head(d1)

class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=4.0, bce_weight=0.45, smooth=1e-5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], dtype=torch.float32))
        self.bce_weight = bce_weight
        self.smooth = smooth

    def forward(self, logits, target):
        pw = self.bce.pos_weight.to(logits.device)
        self.bce.pos_weight = pw
        bce = self.bce(logits, target)
        p = torch.sigmoid(logits).reshape(logits.size(0), -1)
        t = target.reshape(target.size(0), -1)
        inter = (p*t).sum(dim=1)
        den = p.sum(dim=1) + t.sum(dim=1)
        dice = 1.0 - ((2.0*inter + self.smooth) / (den + self.smooth))
        dice = dice.mean()
        return self.bce_weight * bce + (1.0 - self.bce_weight) * dice


In [7]:
train_df = pd.read_csv(TRAIN_CSV)
train_df["patient_id"] = train_df["patient_id"].astype(int)

all_train_imgs = sorted(TRAIN_IMG_DIR.glob("*.nii.gz"))
all_train_msks = [TRAIN_MASK_DIR / p.name for p in all_train_imgs]
all_test_imgs = sorted(TEST_IMG_DIR.glob("*.nii.gz"))

print("train samples:", len(all_train_imgs), "test samples:", len(all_test_imgs))
assert len(all_train_imgs) == len(all_train_msks)
for p in all_train_msks[:3]:
    assert p.exists(), f"missing mask: {p}"


train samples: 49 test samples: 10


In [8]:
CFG = {
    "n_splits": 5,
    "epochs": 60,
    "batch_size": 2,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "num_workers": 0,
    "base_ch": 24,
    "grad_clip": 1.0,
    "tta": True,
    "early_stop_patience": 12,
}

print(CFG)


{'n_splits': 5, 'epochs': 36, 'batch_size': 2, 'lr': 0.001, 'weight_decay': 0.0001, 'num_workers': 0, 'base_ch': 24, 'grad_clip': 1.0, 'tta': True}


In [10]:
def infer_prob_tta(model, img_1dchw):
    # img shape: (1,1,D,H,W)
    model.eval()
    probs = []
    with torch.no_grad():
        # no flip
        p0 = torch.sigmoid(model(img_1dchw))
        probs.append(p0)

        # flip H
        x = torch.flip(img_1dchw, dims=[3])
        p = torch.sigmoid(model(x))
        p = torch.flip(p, dims=[3])
        probs.append(p)

        # flip W
        x = torch.flip(img_1dchw, dims=[4])
        p = torch.sigmoid(model(x))
        p = torch.flip(p, dims=[4])
        probs.append(p)

        # flip H+W
        x = torch.flip(img_1dchw, dims=[3,4])
        p = torch.sigmoid(model(x))
        p = torch.flip(p, dims=[3,4])
        probs.append(p)

    return torch.mean(torch.stack(probs, dim=0), dim=0)

def train_one_fold(fold, tr_idx, va_idx):
    tr_imgs = [all_train_imgs[i] for i in tr_idx]
    tr_msks = [all_train_msks[i] for i in tr_idx]
    va_imgs = [all_train_imgs[i] for i in va_idx]
    va_msks = [all_train_msks[i] for i in va_idx]

    tr_ds = AneurysmDataset(tr_imgs, tr_msks, augment=True)
    va_ds = AneurysmDataset(va_imgs, va_msks, augment=False)

    tr_loader = DataLoader(tr_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"])
    va_loader = DataLoader(va_ds, batch_size=1, shuffle=False, num_workers=CFG["num_workers"])

    model = ResUNet3D(base=CFG["base_ch"]).to(DEVICE)
    criterion = DiceBCELoss(pos_weight=5.0, bce_weight=0.45)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"], eta_min=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_vs = -1.0
    best_path = CKPT_DIR / f"fold_{fold}.pt"
    no_improve = 0

    for epoch in range(CFG["epochs"]):
        model.train()
        losses = []
        for batch in tr_loader:
            x = batch["image"].to(DEVICE)
            y = batch["mask"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())

        scheduler.step()

        # val: 先用固定阈值0.5快速评估，保存best
        model.eval()
        gts, preds = [], []
        with torch.no_grad():
            for batch in va_loader:
                x = batch["image"].to(DEVICE)
                y = batch["mask"].cpu().numpy()[0,0]  # (D,H,W)
                spacing = batch["spacing"].numpy()[0]
                p = torch.sigmoid(model(x)).cpu().numpy()[0,0]
                pm = (p > 0.5).astype(np.uint8)
                gts.append(volume_from_binary_mask(y, spacing))
                preds.append(volume_from_binary_mask(pm, spacing))
        val_vs = volumetric_similarity(gts, preds)
        print(f"Fold {fold} | Epoch {epoch+1:02d}/{CFG['epochs']} | loss {np.mean(losses):.4f} | valVS@0.5 {val_vs:.4f}")

        if val_vs > best_vs:
            best_vs = val_vs
            torch.save(model.state_dict(), best_path)
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= CFG["early_stop_patience"]:
            print(f"Fold {fold} early stop at epoch {epoch+1}, bestVS={best_vs:.4f}")
            break

    print(f"Fold {fold} best VS = {best_vs:.4f} | {best_path}")
    return best_path

def build_oof_predictions(fold_models, splits):
    # 返回每个样本的概率图(用于阈值/融合搜索)，以及体积记录
    oof = []
    for fold, (_, va_idx) in enumerate(splits):
        model = ResUNet3D(base=CFG["base_ch"]).to(DEVICE)
        model.load_state_dict(torch.load(fold_models[fold], map_location=DEVICE))
        model.eval()

        va_imgs = [all_train_imgs[i] for i in va_idx]
        va_msks = [all_train_msks[i] for i in va_idx]
        va_ds = AneurysmDataset(va_imgs, va_msks, augment=False)
        va_loader = DataLoader(va_ds, batch_size=1, shuffle=False)

        with torch.no_grad():
            for batch in va_loader:
                pid = int(batch["pid"][0]) if isinstance(batch["pid"], torch.Tensor) else int(batch["pid"])
                x = batch["image"].to(DEVICE)
                y = batch["mask"].cpu().numpy()[0,0]
                spacing = batch["spacing"].numpy()[0]

                if CFG["tta"]:
                    p = infer_prob_tta(model, x).cpu().numpy()[0,0]
                else:
                    p = torch.sigmoid(model(x)).cpu().numpy()[0,0]

                oof.append({
                    "pid": pid,
                    "spacing": spacing.astype(np.float32),
                    "gt_mask": y.astype(np.uint8),
                    "prob": p.astype(np.float16),
                })
    return oof


In [11]:
start = time.time()
vol_series = train_df.sort_values("patient_id")["volume"].values
# 分层折分：按体积分箱，减少fold间难度波动
try:
    vol_bins = pd.qcut(vol_series, q=5, labels=False, duplicates="drop")
except Exception:
    vol_bins = pd.cut(vol_series, bins=5, labels=False)

skf = StratifiedKFold(n_splits=CFG["n_splits"], shuffle=True, random_state=SEED)
splits = list(skf.split(np.arange(len(all_train_imgs)), vol_bins))
fold_models = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    model_path = train_one_fold(fold, tr_idx, va_idx)
    fold_models.append(model_path)

print("Trained fold models:")
for p in fold_models:
    print(" -", p)
print(f"Training done in {(time.time()-start)/60:.1f} min")


/var/folders/d7/pl_p_f4x5lb6768z5pnxfv5h0000gn/T/ipykernel_13374/2739695661.py:46: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/var/folders/d7/pl_p_f4x5lb6768z5pnxfv5h0000gn/T/ipykernel_13374/2739695661.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/var/folders/d7/pl_p_f4x5lb6768z5pnxfv5h0000gn/T/ipykernel_13374/2739695661.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Fold 0 | Epoch 01/36 | loss 0.7535 | valVS@0.5 0.1756


/var/folders/d7/pl_p_f4x5lb6768z5pnxfv5h0000gn/T/ipykernel_13374/2739695661.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Fold 0 | Epoch 02/36 | loss 0.6238 | valVS@0.5 0.6345


/var/folders/d7/pl_p_f4x5lb6768z5pnxfv5h0000gn/T/ipykernel_13374/2739695661.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


...

Fold 1 | Epoch 03/36 | loss 0.5975 | valVS@0.5 0.0000


/var/folders/d7/pl_p_f4x5lb6768z5pnxfv5h0000gn/T/ipykernel_13374/2739695661.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Fold 1 | Epoch 04/36 | loss 0.5718 | valVS@0.5 0.3755


/var/folders/d7/pl_p_f4x5lb6768z5pnxfv5h0000gn/T/ipykernel_13374/2739695661.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


KeyboardInterrupt: 

In [ ]:
# ===== OOF自动调参：阈值thr + 软硬融合alpha + 线性校准 =====
oof_items = build_oof_predictions(fold_models, splits)

gt_map = {int(r.patient_id): float(r.volume) for _, r in train_df.iterrows()}

thr_grid = np.arange(0.30, 0.71, 0.02)
alpha_grid = np.arange(0.0, 1.01, 0.1)  # blend = alpha*hard + (1-alpha)*soft

best = {"vs": -1, "thr": 0.5, "alpha": 1.0}
cache = []

for thr in thr_grid:
    hard_pred = []
    soft_pred = []
    y_true = []
    pids = []
    for it in oof_items:
        pid = int(it["pid"])
        spacing = it["spacing"]
        prob = it["prob"].astype(np.float32)
        hard = (prob > thr).astype(np.uint8)
        vh = volume_from_binary_mask(hard, spacing)
        vs = volume_from_prob(prob, spacing)
        hard_pred.append(vh)
        soft_pred.append(vs)
        y_true.append(gt_map[pid])
        pids.append(pid)

    hard_pred = np.asarray(hard_pred, dtype=np.float64)
    soft_pred = np.asarray(soft_pred, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.float64)

    for a in alpha_grid:
        blend = a * hard_pred + (1.0 - a) * soft_pred
        vs = volumetric_similarity(y_true, blend)
        cache.append((vs, thr, a))
        if vs > best["vs"]:
            best = {"vs": float(vs), "thr": float(thr), "alpha": float(a)}

print("Best OOF (before calibration):", best)

# 用最佳thr/alpha产生OOF预测，拟合线性校准 y = k*x + b
oof_rows = []
for it in oof_items:
    pid = int(it["pid"])
    spacing = it["spacing"]
    prob = it["prob"].astype(np.float32)
    hard = (prob > best["thr"]).astype(np.uint8)
    vh = volume_from_binary_mask(hard, spacing)
    vs = volume_from_prob(prob, spacing)
    blend = best["alpha"] * vh + (1.0 - best["alpha"]) * vs
    gt = gt_map[pid]
    oof_rows.append({"patient_id": pid, "gt": gt, "hard": vh, "soft": vs, "blend": blend})

oof_df = pd.DataFrame(oof_rows).sort_values("patient_id").reset_index(drop=True)
k, b = np.polyfit(oof_df["blend"].values, oof_df["gt"].values, deg=1)
oof_df["calibrated"] = np.clip(k * oof_df["blend"].values + b, 0, None)
vs_uncal = volumetric_similarity(oof_df["gt"].values, oof_df["blend"].values)
vs_cal = volumetric_similarity(oof_df["gt"].values, oof_df["calibrated"].values)
use_calibration = bool(vs_cal >= vs_uncal)

print(f"OOF VS uncalibrated: {vs_uncal:.6f}")
print(f"OOF VS calibrated  : {vs_cal:.6f}")
print("use_calibration:", use_calibration, "k:", round(float(k), 6), "b:", round(float(b), 6))

oof_df.to_csv(PRED_DIR / "oof_predictions_v2.csv", index=False)


In [ ]:
# ===== 测试推理：fold集成 + TTA + 最优体积策略 + 校准 =====
models = []
for mp in fold_models:
    m = ResUNet3D(base=CFG["base_ch"]).to(DEVICE)
    m.load_state_dict(torch.load(mp, map_location=DEVICE))
    m.eval()
    models.append(m)

rows = []
blend_rows = []

for p in tqdm(all_test_imgs, desc="Test inference"):
    pid = parse_pid(p)
    arr, spacing = load_nii(p)
    arr = robust_zscore(arr)
    arr = np.transpose(arr, (2,0,1)).astype(np.float32)
    x = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(DEVICE)

    fold_probs = []
    with torch.no_grad():
        for m in models:
            if CFG["tta"]:
                prob = infer_prob_tta(m, x).cpu().numpy()[0,0]
            else:
                prob = torch.sigmoid(m(x)).cpu().numpy()[0,0]
            fold_probs.append(prob)

    mean_prob = np.mean(fold_probs, axis=0)
    hard = (mean_prob > best["thr"]).astype(np.uint8)
    vh = volume_from_binary_mask(hard, spacing)
    vs = volume_from_prob(mean_prob, spacing)
    blend = best["alpha"] * vh + (1.0 - best["alpha"]) * vs
    pred = np.clip(k * blend + b, 0, None) if use_calibration else max(blend, 0.0)

    rows.append({"patient_id": pid, "volume": float(pred)})
    blend_rows.append({
        "patient_id": pid,
        "hard_volume": float(vh),
        "soft_volume": float(vs),
        "blend_volume": float(blend),
        "final_volume": float(pred),
    })

sub_df = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
blend_df = pd.DataFrame(blend_rows).sort_values("patient_id").reset_index(drop=True)

# 实验目录输出
exp_sub = PRED_DIR / "submission.csv"
exp_blend = PRED_DIR / "blend_details.csv"
sub_df.to_csv(exp_sub, index=False)
blend_df.to_csv(exp_blend, index=False)

# 平台标准输出
platform_paths = [
    OUTPUT_ROOT / "submission.csv",
    Path("/root/setup/solution/working/submission.csv"),
    Path("/working/submission.csv"),
]
saved_paths = []
for pp in platform_paths:
    try:
        pp.parent.mkdir(parents=True, exist_ok=True)
        sub_df.to_csv(pp, index=False)
        saved_paths.append(str(pp))
    except Exception as e:
        print("skip", pp, "reason:", e)

meta = {
    "run_name": RUN_NAME,
    "base_dir": str(BASE_DIR),
    "output_root": str(OUTPUT_ROOT),
    "saved_submission_paths": saved_paths,
    "best_oof": best,
    "calibration": {"use": use_calibration, "k": float(k), "b": float(b), "oof_vs_uncal": float(vs_uncal), "oof_vs_cal": float(vs_cal)},
    "cfg": CFG,
}
with open(LOG_DIR / "run_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved experiment submission:", exp_sub)
print("Saved platform submission paths:")
for p in saved_paths:
    print(" -", p)
print(sub_df.head())


## 提分建议(继续迭代)
- 先将 `epochs` 提升到 `50~80`，通常比换更大模型更有效。
- 尝试 `base_ch=28`（显存允许时）和 `batch_size=1`。
- 可缩小阈值网格到 `0.36~0.56`、步长 `0.01` 做更细调参。
- 保留当前的 OOF 校准，它对体积指标通常有稳定增益。


## 为什么v3反而降分
- 小样本(49例)下，`epochs=100 + 更大通道`容易拟合训练噪声。
- `batch_size=1` 与 BatchNorm3d 组合通常不稳定，验证波动变大。
- 阈值细网格本身不是问题，问题在于上游分割概率质量下降。
